<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/mlflow_gateway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLflow AI Gateway

[MLflow AI Gateway](https://mlflow.org/docs/latest/llms/gateway/index.html) is a unified LLM proxy built into the MLflow tracking server. It lets you connect multiple LLM providers behind a single endpoint with built-in features like:

- **Multi-provider routing** — OpenAI, Anthropic, Gemini, Mistral, Bedrock, Ollama, and more
- **Secrets management** — provider API keys stored encrypted in the server, not in client code
- **Budget tracking** — per-user or per-endpoint token budgets
- **Fallback & retry** — automatic failover to backup models
- **Traffic splitting** — route a percentage of requests to different models (A/B testing)
- **OpenAI-compatible API** — works with any OpenAI SDK client
- **Usage tracing** — every LLM call logged as an MLflow trace automatically

The gateway is configured through the MLflow UI and runs as part of `mlflow server` (MLflow ≥ 3.0). See the [MLflow AI Gateway documentation](https://mlflow.org/docs/latest/llms/gateway/index.html) for full details.

## Installation

If you're opening this Notebook on Colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-llms-openai-like

In [ ]:
!pip install llama-index mlflow

## Prerequisites: Start MLflow Server

The MLflow AI Gateway is included in the MLflow tracking server. Start it with:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

Then open the MLflow UI at [http://localhost:5000](http://localhost:5000).

## Create a Gateway Endpoint

Before making LLM requests, you need to create a **gateway endpoint** — a named route that maps to a specific LLM provider and model.

The easiest way is through the MLflow UI:

1. Open the MLflow UI at [http://localhost:5000](http://localhost:5000)
2. Navigate to **AI Gateway** in the left sidebar
3. Click **Create Endpoint**
4. Give your endpoint a name (e.g., `my-chat-endpoint`)
5. Select a provider (e.g., OpenAI) and model (e.g., `gpt-4o-mini`)
6. Enter your provider API key — it is stored encrypted on the server
7. Click **Save**

![Create Endpoint UI](mlflow_gateway_images/create_endpoint.png)

See the [MLflow AI Gateway documentation](https://mlflow.org/docs/latest/llms/gateway/index.html) for advanced setup options including programmatic endpoint creation via the REST API.

In [ ]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.llms import ChatMessage

MLFLOW_GATEWAY_URL = "http://localhost:5000"
ENDPOINT_NAME = "my-chat-endpoint"  # the endpoint you created above

llm = OpenAILike(
    model=ENDPOINT_NAME,
    api_base=f"{MLFLOW_GATEWAY_URL}/gateway/openai/v1",
    api_key="unused",  # the gateway manages provider keys; this field is not validated
    is_chat_model=True,
    context_window=4096,
)

## Call `chat` with a ChatMessage List

In [ ]:
messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="What is MLflow AI Gateway?"),
]
resp = llm.chat(messages)
print(resp)

### Streaming Chat

In [ ]:
messages = [
    ChatMessage(role="user", content="Tell me a short story about a robot."),
]
resp = llm.stream_chat(messages)
for chunk in resp:
    print(chunk.delta, end="", flush=True)

## Call `complete` with a Prompt

In [ ]:
resp = llm.complete("Explain the benefits of using a LLM gateway in one paragraph.")
print(resp)

### Streaming Complete

In [ ]:
resp = llm.stream_complete("List 5 LLM providers supported by MLflow Gateway.")
for chunk in resp:
    print(chunk.delta, end="", flush=True)

## Use in a LlamaIndex Pipeline

The `llm` object is a standard LlamaIndex LLM and can be used anywhere in a pipeline — as a query engine, a chat engine, or with agents.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings

# Set the LLM globally for all LlamaIndex operations
Settings.llm = llm

# Build an index and query it
# documents = SimpleDirectoryReader("./data").load_data()
# index = VectorStoreIndex.from_documents(documents)
# query_engine = index.as_query_engine()
# response = query_engine.query("What is in these documents?")
# print(response)

## Alternative: Query the Gateway Directly

You can also call gateway endpoints directly — without LlamaIndex — using the OpenAI SDK or plain HTTP requests.

In [ ]:
# Using the OpenAI SDK (recommended)
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:5000/gateway/openai/v1",
    api_key="unused",  # provider keys are managed server-side
)

response = client.chat.completions.create(
    model="my-chat-endpoint",  # your gateway endpoint name
    messages=[{"role": "user", "content": "What is MLflow AI Gateway?"}],
)
print(response.choices[0].message.content)

In [ ]:
# Or using plain HTTP requests (MLflow Invocations API)
import requests

response = requests.post(
    "http://localhost:5000/gateway/my-chat-endpoint/mlflow/invocations",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is MLflow AI Gateway?"},
        ],
        "temperature": 0.7,
    },
)
print(response.json()["choices"][0]["message"]["content"])

## Gateway Features

The features below are configured in the MLflow UI — **no changes to your LlamaIndex code are needed**.

### Fallback & Retry

Configure a primary model and one or more fallback models on an endpoint. If the primary model fails or is rate-limited, the gateway automatically retries with the next fallback. Set this up in the MLflow UI under **AI Gateway → Edit Endpoint → Fallback Configuration**.

### Traffic Splitting

Route a percentage of requests to different models for A/B testing. For example, 90% to `gpt-4o-mini` and 10% to `gpt-4o`. Configure via **AI Gateway → Edit Endpoint → Routing Strategy**.

### Budget Tracking

Set token or cost budgets per endpoint or per user. When the budget is exhausted, the gateway returns an error. Configure via **AI Gateway → Budgets**.

![Budget Tracking UI](mlflow_gateway_images/budget_tracking.png)

### Usage Tracing

Enable `usage_tracking` on an endpoint to automatically log every LLM call as an MLflow trace in an experiment. This gives you a full audit trail of inputs, outputs, latencies, and token usage with zero instrumentation code.

The **Usage dashboard** shows request volume, latency, and error rates at a glance:

![Usage Dashboard](mlflow_gateway_images/usage_dashboard.png)

The **Logs tab** lists every traced request with its response, token counts, execution time, and status:

![Usage Traces](mlflow_gateway_images/usage_traces.png)

Click any trace to see the full **request and response detail**, including the exact input prompt and model output:

![Trace Detail](mlflow_gateway_images/usage_trace_detail.png)